In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

In [2]:
from numba import njit, int64, float64
from numba.experimental import jitclass
from numba.typed import List
from numba import types
import numpy as np


# PAIR_TYPE = types.Tuple((float64, int64))
TUPLE_TYPE = types.Tuple((float64, int64))

heap_spec = [
    ('data', types.ListType(TUPLE_TYPE))
]

@jitclass(heap_spec)
class Heap:

    def __init__(self):
        self.data = List.empty_list(TUPLE_TYPE)

    def push(self, key, idx):
        self.data.append((key, idx))
        i = len(self.data) - 1
        while i > 0:
            parent = (i - 1) // 2
            if self.data[i][0] <= self.data[parent][0]:
                break
            self.data[i], self.data[parent] = self.data[parent], self.data[i]
            i = parent

    def pop(self):
        top = self.data[0]
        last = self.data.pop()
        if len(self.data) == 0:
            return top
        self.data[0] = last
        i = 0
        while True:
            left = 2 * i + 1
            right = 2 * i + 2
            largest = i
            if left < len(self.data) and self.data[left][0] > self.data[largest][0]:
                largest = left
            if right < len(self.data) and self.data[right][0] > self.data[largest][0]:
                largest = right
            if largest == i:
                break
            self.data[i], self.data[largest] = self.data[largest], self.data[i]
            i = largest
        return top
    
heap = Heap()
heap.push(-1.0, 0)
heap.pop()

(-1.0, 0)

In [3]:
import numpy as np

from numba.experimental import jitclass
from numba.types import int64, float64
from numba.typed import List
from numba import njit

from optikon import Propositionalization, compute_bounds, equal_width_propositionalization, full_propositionalization
from testdata import mvn_with_correlation

@jitclass
class CanonicalTreeSearchNode:

    key: int64[:]
    critical: int64[:]
    remaining: int64[:]
    support: int64[:]

    def __init__(self, key, critical, remaining, support):
        self.key = key
        self.critical = critical
        self.remaining =remaining
        self.support = support

@njit
def make_root(x, prop):
    l, u = compute_bounds(x)
    remaining = prop.nontrivial(l, u, np.arange(len(prop)))
    empty = np.empty(0, dtype=np.int64)
    return CanonicalTreeSearchNode(empty, empty, remaining, np.arange(len(x)))

def node_to_string(node):
    return f'Node({node.key}, {node.value}, {node.bound})'

# @njit
# def dummy_obj(node, x, y):
#     return float64(len(node.key))

@jitclass
class KeyLength:

    def __init__(self):
        pass

    def compute(self, node, x, y):
        return float64(len(node.key))
    
key_length = KeyLength()
    
@jitclass
class KeyLengthBound:

    def __init__(self):
        pass

    def compute(self, node, x, y):
        return float(len(node.key) + len(node.remaining))
    
key_length_bound = KeyLengthBound()

# @njit
# def dummy_bnd(node, x, y):
#     return float64(len(node.key) + len(node.remaining))

# @njit
# def weighted_support(node, x, y):
#     return y[node.support].sum()

@jitclass
class WeightedSupport:

    def __init__(self):
        pass

    def compute(self, node, x, y):
        return y[node.support].sum()

weighted_support = WeightedSupport()

# @njit
# def weighted_support_bound_naive(node, x, y):
#     pos = (y > 0)
#     return y[node.support][pos].sum() 

@jitclass
class WeightedSupportBoundNaive:

    def __init__(self):
        pass

    def compute(self, node, x, y):
        pos = (y > 0)
        return y[node.support[pos[node.support]]].sum()
    
weighted_support_bound_naive = WeightedSupportBoundNaive()


CanonicalTreeSearchNodeType = CanonicalTreeSearchNode.class_type.instance_type

@jitclass
class SearchSpec:

    x: float64[:, :]
    y: float64[:]
    prop: Propositionalization
    max_depth: int64

    def __init__(self, x, y, prop, max_depth=4):
        self.x = x
        self.y = y
        self.prop = prop
        self.max_depth = max_depth
    
@njit
def refinement(node, search_spec):
    res = List()
    for p_idx in range(len(node.remaining)):
        p = node.remaining[p_idx]

        _key = np.empty(len(node.key) + 1, dtype=np.int64)
        _key[:-1] = node.key
        _key[-1] = p
        _sup = node.support[search_spec.prop.support(p, search_spec.x[node.support])]
        
        _crit = np.empty(len(node.critical) + p_idx, dtype=np.int64)
        _crit[:len(node.critical)] = node.critical
        _crit[len(node.critical):] = node.remaining[:p_idx]
        l, u = compute_bounds(search_spec.x[_sup])
        if len(search_spec.prop.trivial(l, u, _crit))>0:
            continue

        _remaining = search_spec.prop.nontrivial(l, u, node.remaining[p_idx+1:])
        res.append(CanonicalTreeSearchNode(_key, _crit, _remaining, _sup))
    return res

@jitclass
class SearchResult:

    best: CanonicalTreeSearchNode
    value: float64
    bound: float64
    nodes_created: int64
    edges_tested: int64

    def __init__(self, best, value, nodes_created, edges_tested):
        self.best = best
        self.value = value
        self.nodes_created = nodes_created
        self.edges_tested = edges_tested

    def __str__(self):
        return self.best.key, self.value


@njit
def run(spec, obj, bnd):
    # if obj is None:
    #     obj = key_length
    # if bnd is None:
    #     bnd = key_length_bound

    heap = Heap()
    nodes = List.empty_list(CanonicalTreeSearchNodeType) #List.empty_list(CanonicalTreeSearchNode.class_type.instance_type)
    freelist = List.empty_list(int64)

    root = make_root(spec.x, spec.prop)
    nodes.append(root)
    heap.push(-bnd.compute(root, spec.x, spec.y), 0)

    best = root
    best_value = obj.compute(root, spec.x, spec.y)
    created = 1
    non_canonical = 0

    while len(heap.data) > 0:
        neg_bound, idx = heap.pop()
        node = nodes[idx]
        freelist.append(idx)

        if -neg_bound < best_value:
            continue
        if len(node.key) >= spec.max_depth:
            continue

        children = refinement(node, spec)
        created += len(children)
        non_canonical += len(node.remaining) - len(children)

        for child in children:
            val = obj.compute(child, spec.x, spec.y)
            bound = bnd.compute(child, spec.x, spec.y)
            if val > best_value:
                best = child
                best_value = val

            if len(freelist) > 0:
                reuse_idx = freelist.pop()
                nodes[reuse_idx] = child
                heap.push(-bound, reuse_idx)
            else:
                nodes.append(child)
                heap.push(-bound, len(nodes) - 1)

    return SearchResult(best, best_value, created, created-1+non_canonical)
    

x = mvn_with_correlation(100, seed=0)
y = np.random.default_rng(seed=0).normal(size=100)
prop = equal_width_propositionalization(x) # full_propositionalization(x) #
search = SearchSpec(x, y, prop, 8)
run(search, key_length, key_length_bound).best.key

array([ 4, 15, 22, 33, 39, 47, 56, 65])

In [4]:
%timeit run(search, key_length, key_length_bound).best.key

940 ms ± 9.59 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
x = mvn_with_correlation(100, seed=0)
y = np.random.default_rng(seed=0).normal(size=100)
prop = equal_width_propositionalization(x) # full_propositionalization(x) #
search = SearchSpec(x, y, prop, 8)
res = run(search, weighted_support, weighted_support_bound_naive)
res.best.key, prop.str_from_conj(res.best.key), res.value

(array([ 8, 16, 23, 49, 59, 66]),
 'x1 >= -1.472 & x1 <= 1.740 & x2 >= 0.612 & x3 <= 1.730 & x4 >= -2.580 & x4 <= 0.538',
 20.397495605985817)

In [11]:
res = run(search, weighted_support, weighted_support_bound_naive)
res.best.key, prop.str_from_conj(res.best.key), res.value

(array([ 8, 16, 23, 49, 59, 66]),
 'x1 >= -1.472 & x1 <= 1.740 & x2 >= 0.612 & x3 <= 1.730 & x4 >= -2.580 & x4 <= 0.538',
 20.397495605985817)